In [ ]:
# Run this cell first to install required packages in JupyterLite
%pip install -q numpy matplotlib ipywidgets scikit-learn


# Figure 16.12: Activation Histogram with Outliers
**Xavier/Glorot init:** $W \sim \mathcal{N}\!\left(0, \frac{2}{n_{in}+n_{out}}\right)$ — preserves variance for sigmoid/tanh.

**Kaiming/He init:** $W \sim \mathcal{N}\!\left(0, \frac{2}{n_{in}}\right)$ — accounts for ReLU's half-rectification.

Poor initialization leads to:
- **Vanishing signals** (small random / sigmoid without Xavier)
- **Exploding signals** (large random)
- **Dead neurons** (zeros / ReLU without Kaiming)
- **Symmetry breaking failure** (all-zeros weights)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def init_weights(scheme, fan_in, fan_out, rng):
    if scheme == 'zeros':
        return np.zeros((fan_in, fan_out))
    elif scheme == 'large':
        return rng.normal(0, 2.0, (fan_in, fan_out))
    elif scheme == 'small':
        return rng.normal(0, 0.01, (fan_in, fan_out))
    elif scheme == 'xavier':
        return rng.normal(0, np.sqrt(2/(fan_in+fan_out)), (fan_in, fan_out))
    else:  # kaiming
        return rng.normal(0, np.sqrt(2/fan_in), (fan_in, fan_out))


def activate(z, act):
    if act == 'sigmoid': return 1/(1+np.exp(-np.clip(z,-20,20)))
    if act == 'tanh':    return np.tanh(z)
    if act == 'relu':    return np.maximum(0, z)
    if act == 'leakyrelu': return np.where(z>0, z, 0.01*z)
    return z


def act_deriv(z, act):
    if act == 'sigmoid': s=1/(1+np.exp(-np.clip(z,-20,20))); return s*(1-s)
    if act == 'tanh':    return 1 - np.tanh(z)**2
    if act == 'relu':    return (z > 0).astype(float)
    if act == 'leakyrelu': return np.where(z>0, 1.0, 0.01)
    return np.ones_like(z)


def draw_playground(scheme='xavier', act='tanh', depth=6, width=64):
    rng = np.random.default_rng(42)
    batch = 64
    Ws = [init_weights(scheme, width, width, rng) for _ in range(depth)]

    # Forward pass
    h = rng.normal(0, 1, (batch, width))
    pre_acts, post_acts = [], [h]
    for l in range(depth):
        z = h @ Ws[l]
        pre_acts.append(z)
        h = activate(z, act)
        post_acts.append(h)

    # Backward pass
    delta = post_acts[-1] / batch
    grad_norms = []
    for l in range(depth-1, -1, -1):
        dz = delta * act_deriv(pre_acts[l], act)
        grad_norms.insert(0, np.linalg.norm(dz) / np.sqrt(batch*width))
        delta = dz @ Ws[l].T

    # Stats
    act_stds = [np.std(h) for h in post_acts]
    dead_fracs = [np.mean(np.abs(h) < 1e-4) for h in post_acts[1:]]
    layer_names = ['Input'] + [f'L{i+1}' for i in range(depth)]

    def color(std, dead):
        if std < 0.01 or dead > 0.8: return '#dc2626'
        if std > 5 or dead > 0.4: return '#d97706'
        return '#059669'

    bar_colors = [color(s, dead_fracs[i-1] if i>0 else 0) for i,s in enumerate(act_stds)]
    grad_colors = ['#dc2626' if g<1e-6 else '#d97706' if g>10 else '#7c3aed' for g in grad_norms]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 1. Activation std per layer
    axes[0].bar(layer_names, act_stds, color=bar_colors)
    axes[0].axhline(1.0, color='#059669', ls='--', lw=1.5, label='σ=1 (ideal)')
    axes[0].set_yscale('log'); axes[0].set_ylabel('Activation std (log)')
    axes[0].set_title('Activation std per layer', fontsize=11)
    axes[0].legend(fontsize=9); axes[0].grid(True, which='both', alpha=0.3)
    axes[0].tick_params(axis='x', rotation=45)

    # 2. Gradient norms
    axes[1].bar([f'L{i+1}' for i in range(depth)], grad_norms, color=grad_colors)
    axes[1].set_yscale('log'); axes[1].set_ylabel('Gradient norm (log)')
    axes[1].set_title('Gradient norm per layer', fontsize=11)
    axes[1].grid(True, which='both', alpha=0.3)
    axes[1].tick_params(axis='x', rotation=45)

    # 3. Activation histograms (first, middle, last)
    show_layers = [1, depth//2+1, depth]
    hist_colors = ['#2563eb', '#7c3aed', '#059669']
    for li, hc in zip(show_layers, hist_colors):
        vals = post_acts[li].flatten()
        axes[2].hist(vals, bins=40, alpha=0.55, color=hc, label=f'Layer {li}')
    axes[2].set_xlabel('Activation value'); axes[2].set_ylabel('Count')
    axes[2].set_title('Activation histograms (3 layers)', fontsize=11)
    axes[2].legend(fontsize=9)

    # Diagnosis
    last_std = act_stds[-1]; max_dead = max(dead_fracs)*100
    if scheme == 'zeros': diag = '⛔ ZEROS: symmetry broken — all neurons identical, no learning'
    elif last_std < 0.01: diag = '⛔ VANISHING: activations → 0, gradients vanish'
    elif last_std > 5:   diag = '⚠ EXPLODING: activations too large, training unstable'
    elif max_dead > 50:  diag = '⚠ DEAD NEURONS: ReLU collapse, use Kaiming or Leaky ReLU'
    else:                diag = '✓ HEALTHY: stable signal and gradients throughout'

    plt.suptitle(f'{scheme} init | {act} | depth={depth} | width={width}\n{diag}',
                 fontsize=10, y=0.0)
    plt.tight_layout()
    plt.show()
    print(f'Final layer: act_std={act_stds[-1]:.4f}  dead={max_dead:.1f}%')
    print(f'Gradient L1 norm: {grad_norms[0]:.6f}')


widgets.interact(
    draw_playground,
    scheme=widgets.Dropdown(
        options=[('Zeros (broken)', 'zeros'), ('Large random σ=2.0', 'large'),
                 ('Small random σ=0.01', 'small'), ('Xavier / Glorot', 'xavier'),
                 ('Kaiming / He', 'kaiming')],
        value='xavier', description='Init scheme'),
    act=widgets.Dropdown(
        options=[('Sigmoid','sigmoid'),('Tanh','tanh'),('ReLU','relu'),('Leaky ReLU','leakyrelu')],
        value='tanh', description='Activation'),
    depth=widgets.IntSlider(min=3, max=10, step=1, value=6, description='Depth', continuous_update=False),
    width=widgets.SelectionSlider(options=[16,32,64,128], value=64, description='Width', continuous_update=False),
)